### Genre Prediction

In [30]:
import warnings


warnings.filterwarnings("ignore")

In [54]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import jaccard_score, hamming_loss
from sklearn.multiclass import OneVsRestClassifier

import warnings
import joblib

warnings.filterwarnings("ignore")

# Columns to be used for training
columns = ['avg_red', 'avg_green', 'avg_blue', 'has_face', 'num_faces', 'has_text_top_quarter',
           'has_face_center', 'has_text_bottom_half', 'num_text_lines_bottom_quarter', 'title_in_image',
           'year_in_image', 'rating_in_image', 'largest_text_area_ratio', 'faces_area_ratio',
           'has_central_text', 'std_dev_red', 'std_dev_green', 'std_dev_blue', 'peak_red',
           'peak_green', 'peak_blue', 'has_warm_colors', 'has_cool_colors', 'aspect_ratio']

# Load the dataset
df = pd.read_csv('test.csv')

# Clean the 'Movie_Genre' column
df['Movie_Genre'] = df['Movie_Genre'].apply(lambda x: x.split(','))

# Convert all categorical variables into dummy variables
#X = pd.get_dummies(df.drop(['Movie_Genre', 'Movie_Name'], axis=1))
X = df[columns]
y = df['Movie_Genre']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocess the genres using MultiLabelBinarizer
mlb = MultiLabelBinarizer()
y_train = mlb.fit_transform(y_train)
y_test = mlb.transform(y_test)

# Create a dataframe to store the results of each model
result_df = pd.DataFrame(columns=["Classifier", "Jaccard Score", "Hamming Loss"])

# Create an empty DataFrame to store the genre predictions for each model
predictions_df = pd.DataFrame()

# List of classifiers
classifiers = [
    ("Random Forest", RandomForestClassifier(random_state=42, class_weight='balanced')),
    ("KNN", KNeighborsClassifier()),
    ("SVM", SVC(random_state=42, probability=True, class_weight='balanced'))
]

for clf_name, clf in classifiers:
    clf = OneVsRestClassifier(clf)
    
    # Train and evaluate the classifiers
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_pred_prob = clf.predict_proba(X_test)

    # Save the model
    joblib.dump(clf, f'{clf_name}_model.pkl')

    # Limit predicted genres between 1 and 3
    for i in range(len(y_pred)):
        # Number of genres predicted for the movie
        num_genres = y_pred[i].sum()
        
        # If no genre or more than 3 genres predicted, correct the prediction
        if num_genres == 0 or num_genres > 3:
            top_genres = y_pred_prob[i].argsort()[-3:][::-1]
            y_pred[i] = 0
            y_pred[i, top_genres] = 1

    # Calculate jaccard score and hamming loss
    jaccard = jaccard_score(y_test, y_pred, average='samples')
    hamming = hamming_loss(y_test, y_pred)

    # Append the results to the result dataframe
    result_df = result_df.append({
        "Classifier": clf_name,
        "Jaccard Score": jaccard,
        "Hamming Loss": hamming
    }, ignore_index=True)

    # Get the genre predictions in the original (text) format and strip spaces
    pred_genres = mlb.inverse_transform(y_pred)
    pred_genres = [tuple(s.strip() for s in genres) for genres in pred_genres]

    # Strip spaces from the actual genres
    actual_genres = y.iloc[X_test.index].apply(tuple).values
    actual_genres = [tuple(s.strip() for s in genres) for genres in actual_genres]

    # Store the movie id, actual genres, and predicted genres
    temp_df = pd.DataFrame({
        "Movie_Id": X_test.index,
        "Actual Genres": actual_genres,
        "Predicted Genres ({})".format(clf_name): pred_genres
    })

    # If predictions_df is empty, assign temp_df to it, else merge temp_df with predictions_df
    if predictions_df.empty:
        predictions_df = temp_df
    else:
        predictions_df = pd.merge(predictions_df, temp_df, on=["Movie_Id", "Actual Genres"])

joblib.dump(mlb, 'mlb.pkl')
# Save the results to a csv file
result_df.to_csv('genre_prediction_results.csv', index=False)
predictions_df.to_csv('genre_predictions.csv', index=False)


In [55]:
import joblib

# Load the models
rf_model = joblib.load('Random Forest_model.pkl')
knn_model = joblib.load('KNN_model.pkl')
svm_model = joblib.load('SVM_model.pkl')

# List of models
models = [rf_model, knn_model, svm_model]

# Load the binarizer
mlb = joblib.load('mlb.pkl')

# Specify the path of the image
image_path = 'tt0080442.jpg'

# Extract features from the image
features = extract_features(image_path)

# Make predictions with each model
for model in models:
    pred_bin = model.predict([features])
    pred_genres = mlb.inverse_transform(pred_bin)
    print(f"Predicted genres: {pred_genres[0]}")


NameError: name 'extract_features' is not defined